In [ ]:

import pandas as pd
import numpy as np


def recode_attributes(attribute):
    if attribute == 'Non-Failing Donor':
        return 'NF'
    elif attribute == 'Peripartum cardiomyopathy (PPCM)':
        return 'PPCM'
    elif attribute == 'Dilated cardiomyopathy (DCM)':
        return 'DCM'
    elif attribute == 'Hypertrophic cardiomyopathy (HCM)':
        return 'HCM'
    else:
        return 'W'



biomart_path = "/data/bionets/og86asub/alternet-project/alternet/data/biomart.txt"
biomart = pd.read_csv(biomart_path, sep='\t')

appris_path = "/data/bionets/og86asub/alternet-project/alternet/data/appris_data.appris.txt"
appris_df = pd.read_csv(appris_path, sep='\t')


# sample attributions
sample_information_path = "/data/bionets/og86asub/alternet-project/alternet/data/run_table.csv"
sample_attributes = pd.read_csv(sample_information_path)
sample_attributes = sample_attributes.loc[:, ['Run', 'etiology']]
sample_attributes['eti'] = sample_attributes['etiology'].apply(lambda x: recode_attributes(x))

# TMPs rename columns
ktmp = pd.read_csv("/data/bionets/mi34qaba/data/MAGNet/MAGNet_transcript/MAGNet_kallisto_tpms", sep = '\t')
ktmp = ktmp.rename(columns={'target_id': 'transcript_id'})
names = [x.replace('_est_tpms', '') for x in ktmp.columns]
ktmp.columns = names

# Add gene ID
ktmp = ktmp.merge(biomart.loc[: ,['Transcript stable ID', 'Gene stable ID']], left_on ='transcript_id', right_on ='Transcript stable ID')
ktmp = ktmp.drop(columns=['Transcript stable ID'])
ktmp = ktmp.rename(columns={"Gene stable ID":'gene_id'})

# subset to protein coding transcripts
ktmp = ktmp[ktmp.transcript_id.isin(appris_df[appris_df['Transcript type']=='protein_coding']['Transcript ID'])]


NameError: name 'op' is not defined

In [11]:
tf_list

,A1CF
0,ABCF2
1,ABL1
2,ACAA1
3,ACO1
4,ADARB1
...,...
1791,ZSCAN9
1792,ZSWIM1
1793,ZXDA
1794,ZXDB


In [16]:
import os.path as op
basepath = "/data/bionets/og86asub/alternet-project/alternet/data/"

tf_list_path = op.join(basepath, "allTFs_hg38.txt")
tf_list = pd.read_csv(tf_list_path, header=None )
tf_list.columns = ['TF']
tf_list = tf_list.merge(biomart, left_on='TF', right_on='Gene name')

In [17]:

conditions = ['DCM', 'HCM', 'NF']

for condi in conditions:

    # subset data to condition
    samples = sample_attributes[sample_attributes['eti'] == condi]
    samples = samples['Run'].tolist()

    transcript_data_cp = ktmp.loc[:, ['transcript_id', 'gene_id'] + samples].copy(deep=True)

    nonzero_count = np.count_nonzero(transcript_data_cp.values, axis=1)
    frac = nonzero_count/transcript_data_cp.shape[1]
    transcript_data_cp = transcript_data_cp.iloc[np.where(frac>0.50)[0]]

    transcript_data_cp = transcript_data_cp.reset_index()
    transcript_data_cp =transcript_data_cp.drop(columns=['index'])
    print(transcript_data_cp.shape)
    print(len(transcript_data_cp.gene_id.unique()))
    print(len(transcript_data_cp.transcript_id.unique()))

    print(len(transcript_data_cp[transcript_data_cp.transcript_id.isin(tf_list['Transcript stable ID'])]['gene_id'].unique()))
    print(len(transcript_data_cp[transcript_data_cp.transcript_id.isin(tf_list['Transcript stable ID'])]['transcript_id'].unique()))

    transcript_data_cp.to_csv(f"/data/bionets/og86asub/alternet-project/alternet/data/{condi}_magnet_prefiltered_tpm.tsv", sep = '\t', index = False)


(40993, 168)
15959
40993
1539
4189
(42775, 30)
16116
42775
1549
4409
(41982, 168)
15832
41982
1530
4330


In [21]:
transcript_data_cp

,transcript_id,gene_id,SRR10676821,SRR10676822,SRR10676823,SRR10676824,SRR10676825,SRR10676826,SRR10676827,SRR10676828,...,SRR10677161,SRR10677163,SRR10677168,SRR10677171,SRR10677174,SRR10677175,SRR10677176,SRR10677177,SRR10677179,SRR10677184
0,ENST00000419783,ENSG00000233276,18.182400,23.665400,58.730700,10.415700,43.194400,55.551300,44.604100,29.813400,...,7.469420,53.473300,87.091700,7.828650,142.587000,86.153100,15.702000,33.564000,149.306000,9.579430
1,ENST00000419349,ENSG00000233276,0.292345,0.181541,0.883829,0.243651,0.604892,0.508725,0.322050,0.000000,...,0.663738,0.207567,0.544754,0.424345,1.000220,0.116289,0.400908,0.554897,0.418136,0.433982
2,ENST00000643797,ENSG00000233276,0.000000,0.000000,0.359728,0.000000,0.649254,0.000000,0.210922,0.000000,...,0.000000,0.612003,0.000000,0.000000,1.275920,1.083740,0.161790,0.000000,1.907470,0.000000
3,ENST00000531391,ENSG00000166473,1.374890,0.889535,0.034239,0.571389,0.294203,0.925668,0.805371,2.142800,...,0.526265,0.390488,1.620090,1.165040,1.114420,0.749224,1.136220,2.119570,0.329100,0.828792
4,ENST00000527937,ENSG00000166473,0.583894,0.140139,0.000000,0.225796,0.023858,0.011475,0.082714,0.591058,...,0.098087,0.215930,0.269296,0.264715,0.016233,0.069791,0.477872,0.198170,0.018419,0.252184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41977,ENST00000510091,ENSG00000205482,0.397195,0.139399,0.137387,0.238294,0.456423,0.055239,0.127514,0.330749,...,0.067772,0.278306,0.285876,0.041657,0.033892,0.187877,0.468567,1.001040,0.049527,0.233459
41978,ENST00000515156,ENSG00000250138,0.567227,0.141512,2.370690,0.312244,0.359836,0.368929,0.295145,0.115375,...,0.094829,0.486865,0.655804,0.103113,0.130520,0.358449,0.605390,1.246570,0.133815,0.255137
41979,ENST00000505635,ENSG00000251158,0.065305,0.038983,0.065598,0.075307,0.114760,0.023180,0.050163,0.043236,...,0.042068,0.115614,0.035374,0.039918,0.025277,0.070961,0.015279,0.141391,0.025824,0.035174
41980,ENST00000618416,ENSG00000184616,0.769121,0.759989,0.346327,0.559473,0.161998,0.323150,1.094700,0.756638,...,0.592340,0.360954,0.000000,0.103718,0.098629,0.434144,0.721068,0.476099,0.116454,0.425519


In [ ]:
ENST00000000442,ENSG00000173153,ENSG00000010256
ENSG00000145388,ENSG00000166311
ENST00000374685,ENSG0000020423
ENST00000331483,ENSG00000185624,ENSG00000130309
ENST00000383052,ENSG00000067646,ENSG00000067048

In [26]:
transcript_data_cp_minimal =transcript_data_cp[transcript_data_cp.gene_id.isin(['ENSG00000173153','ENSG00000010256','ENSG00000166311','ENSG0000020423', 'ENSG00000185624', 'ENSG00000130309','ENSG00000067646','ENSG00000067048'])].copy()

In [27]:
transcript_data_cp_minimal.to_csv(f"/data/bionets/og86asub/alternet-project/alternet/data/minimal_{condi}_magnet_prefiltered_tpm.tsv", sep = '\t', index = False)
